<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day5_10_(260605)_WMS_%EB%8C%80%EC%8B%9C%EB%B3%B4%EB%93%9C_%EB%A1%9C%EA%B7%B8%EC%9D%B8%26%ED%9A%8C%EC%9B%90%EA%B0%80%EC%9E%85_%EA%B8%B0%EB%B0%98_%EA%B3%A0%EA%B0%9D_%EA%B4%80%EB%A6%AC%EC%9E%90_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/spring-lab/wms-part3/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-jdbc'
    implementation 'org.springframework.boot:spring-boot-starter-security'

    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
    testImplementation 'org.springframework.security:spring-security-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /content/spring-lab/wms-part3/build.gradle


In [2]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/wms_part3
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.sql.init.mode=always
spring.sql.init.schema-locations=classpath:schema.sql

spring.thymeleaf.cache=false

Overwriting /content/spring-lab/wms-part3/src/main/resources/application.properties


In [3]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/schema.sql

CREATE TABLE IF NOT EXISTS users (
    id BIGINT NOT NULL AUTO_INCREMENT,
    email VARCHAR(100) NOT NULL,
    password_hash VARCHAR(255) NOT NULL,
    name VARCHAR(50) NOT NULL,
    role VARCHAR(30) NOT NULL,
    status VARCHAR(30) NOT NULL DEFAULT 'ACTIVE',
    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at DATETIME NULL,
    PRIMARY KEY (id),
    UNIQUE KEY uk_users_email (email)
);

Writing /content/spring-lab/wms-part3/src/main/resources/schema.sql


In [4]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/domain/AppUser.java

package com.example.wmspart3.domain;

import java.time.LocalDateTime;

public class AppUser {

    private Long id;
    private String email;
    private String passwordHash;
    private String name;
    private String role;
    private String status;
    private LocalDateTime createdAt;
    private LocalDateTime updatedAt;

    public AppUser(Long id, String email, String passwordHash, String name,
                   String role, String status, LocalDateTime createdAt, LocalDateTime updatedAt) {
        this.id = id;
        this.email = email;
        this.passwordHash = passwordHash;
        this.name = name;
        this.role = role;
        this.status = status;
        this.createdAt = createdAt;
        this.updatedAt = updatedAt;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPasswordHash() {
        return passwordHash;
    }

    public String getName() {
        return name;
    }

    public String getRole() {
        return role;
    }

    public String getStatus() {
        return status;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }

    public LocalDateTime getUpdatedAt() {
        return updatedAt;
    }

    public boolean isActive() {
        return "ACTIVE".equals(status);
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/domain/AppUser.java


In [5]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/dto/SignupForm.java

package com.example.wmspart3.dto;

public class SignupForm {

    private String email;
    private String password;
    private String name;
    private String role;

    public SignupForm() {
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public String getRole() {
        return role;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setPassword(String password) {
        this.password = password;
    }

    public void setName(String name) {
        this.name = name;
    }

    public void setRole(String role) {
        this.role = role;
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/dto/SignupForm.java


In [6]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/repository/UserRepository.java

package com.example.wmspart3.repository;

import com.example.wmspart3.domain.AppUser;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.stereotype.Repository;

import java.sql.Timestamp;
import java.util.List;
import java.util.Optional;

@Repository
public class UserRepository {

    private final JdbcTemplate jdbcTemplate;

    public UserRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    public void save(String email, String passwordHash, String name, String role) {
        String sql = """
                INSERT INTO users
                (email, password_hash, name, role, status)
                VALUES (?, ?, ?, ?, 'ACTIVE')
                """;

        jdbcTemplate.update(sql, email, passwordHash, name, role);
    }

    public Optional<AppUser> findByEmail(String email) {
        String sql = """
                SELECT id, email, password_hash, name, role, status, created_at, updated_at
                FROM users
                WHERE email = ?
                """;

        List<AppUser> users = jdbcTemplate.query(sql, (rs, rowNum) -> {
            Timestamp updatedAt = rs.getTimestamp("updated_at");

            return new AppUser(
                    rs.getLong("id"),
                    rs.getString("email"),
                    rs.getString("password_hash"),
                    rs.getString("name"),
                    rs.getString("role"),
                    rs.getString("status"),
                    rs.getTimestamp("created_at").toLocalDateTime(),
                    updatedAt == null ? null : updatedAt.toLocalDateTime()
            );
        }, email);

        return users.stream().findFirst();
    }

    public boolean existsByEmail(String email) {
        String sql = "SELECT COUNT(*) FROM users WHERE email = ?";
        Integer count = jdbcTemplate.queryForObject(sql, Integer.class, email);
        return count != null && count > 0;
    }

    public long countAll() {
        String sql = "SELECT COUNT(*) FROM users";
        Long count = jdbcTemplate.queryForObject(sql, Long.class);
        return count == null ? 0 : count;
    }

    public long countByRole(String role) {
        String sql = "SELECT COUNT(*) FROM users WHERE role = ?";
        Long count = jdbcTemplate.queryForObject(sql, Long.class, role);
        return count == null ? 0 : count;
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/repository/UserRepository.java


In [7]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/service/UserService.java

package com.example.wmspart3.service;

import com.example.wmspart3.dto.SignupForm;
import com.example.wmspart3.repository.UserRepository;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Service;

@Service
public class UserService {

    private final UserRepository userRepository;
    private final PasswordEncoder passwordEncoder;

    public UserService(UserRepository userRepository, PasswordEncoder passwordEncoder) {
        this.userRepository = userRepository;
        this.passwordEncoder = passwordEncoder;
    }

    public void signup(SignupForm form) {
        validateSignupForm(form);

        String email = form.getEmail().trim();
        String name = form.getName().trim();
        String role = form.getRole();

        if (userRepository.existsByEmail(email)) {
            throw new IllegalArgumentException("이미 사용 중인 이메일입니다.");
        }

        String passwordHash = passwordEncoder.encode(form.getPassword());

        userRepository.save(email, passwordHash, name, role);
    }

    public long countAllUsers() {
        return userRepository.countAll();
    }

    public long countCustomers() {
        return userRepository.countByRole("ROLE_CUSTOMER");
    }

    public long countAdmins() {
        return userRepository.countByRole("ROLE_ADMIN");
    }

    private void validateSignupForm(SignupForm form) {
        if (isBlank(form.getEmail())) {
            throw new IllegalArgumentException("이메일을 입력해야 합니다.");
        }

        if (!form.getEmail().contains("@")) {
            throw new IllegalArgumentException("이메일 형식이 올바르지 않습니다.");
        }

        if (isBlank(form.getPassword())) {
            throw new IllegalArgumentException("비밀번호를 입력해야 합니다.");
        }

        if (form.getPassword().length() < 4) {
            throw new IllegalArgumentException("비밀번호는 4자 이상이어야 합니다.");
        }

        if (isBlank(form.getName())) {
            throw new IllegalArgumentException("이름을 입력해야 합니다.");
        }

        if (!"ROLE_CUSTOMER".equals(form.getRole()) && !"ROLE_ADMIN".equals(form.getRole())) {
            throw new IllegalArgumentException("사용자 유형을 선택해야 합니다.");
        }
    }

    private boolean isBlank(String value) {
        return value == null || value.trim().isEmpty();
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/service/UserService.java


In [8]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/service/CustomUserDetailsService.java

package com.example.wmspart3.service;

import com.example.wmspart3.domain.AppUser;
import com.example.wmspart3.repository.UserRepository;
import org.springframework.security.core.authority.SimpleGrantedAuthority;
import org.springframework.security.core.userdetails.User;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class CustomUserDetailsService implements UserDetailsService {

    private final UserRepository userRepository;

    public CustomUserDetailsService(UserRepository userRepository) {
        this.userRepository = userRepository;
    }

    @Override
    public UserDetails loadUserByUsername(String email) throws UsernameNotFoundException {
        AppUser appUser = userRepository.findByEmail(email)
                .orElseThrow(() -> new UsernameNotFoundException("사용자를 찾을 수 없습니다."));

        if (!appUser.isActive()) {
            throw new UsernameNotFoundException("비활성 사용자입니다.");
        }

        return new User(
                appUser.getEmail(),
                appUser.getPasswordHash(),
                List.of(new SimpleGrantedAuthority(appUser.getRole()))
        );
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/service/CustomUserDetailsService.java


In [9]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/config/SecurityConfig.java

package com.example.wmspart3.config;

import com.example.wmspart3.service.CustomUserDetailsService;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.core.Authentication;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;
import org.springframework.security.web.authentication.AuthenticationSuccessHandler;

import java.io.IOException;

@Configuration
public class SecurityConfig {

    private final CustomUserDetailsService customUserDetailsService;

    public SecurityConfig(CustomUserDetailsService customUserDetailsService) {
        this.customUserDetailsService = customUserDetailsService;
    }

    @Bean
    public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
        http
                .userDetailsService(customUserDetailsService)
                .authorizeHttpRequests(auth -> auth
                        .requestMatchers("/style.css", "/login", "/signup").permitAll()
                        .requestMatchers("/admin/**").hasRole("ADMIN")
                        .requestMatchers("/customer/**").hasRole("CUSTOMER")
                        .anyRequest().authenticated()
                )
                .formLogin(form -> form
                        .loginPage("/login")
                        .loginProcessingUrl("/login")
                        .usernameParameter("email")
                        .passwordParameter("password")
                        .successHandler(roleBasedSuccessHandler())
                        .failureUrl("/login?error")
                        .permitAll()
                )
                .logout(logout -> logout
                        .logoutUrl("/logout")
                        .logoutSuccessUrl("/login?logout")
                        .invalidateHttpSession(true)
                        .deleteCookies("JSESSIONID")
                )
                .exceptionHandling(exception -> exception
                        .accessDeniedPage("/login?denied")
                );

        return http.build();
    }

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }

    private AuthenticationSuccessHandler roleBasedSuccessHandler() {
        return (request, response, authentication) -> {
            if (hasRole(authentication, "ROLE_ADMIN")) {
                response.sendRedirect("/admin/dashboard");
                return;
            }

            if (hasRole(authentication, "ROLE_CUSTOMER")) {
                response.sendRedirect("/customer/dashboard");
                return;
            }

            response.sendRedirect("/login");
        };
    }

    private boolean hasRole(Authentication authentication, String role) {
        return authentication.getAuthorities().stream()
                .anyMatch(authority -> role.equals(authority.getAuthority()));
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/config/SecurityConfig.java


In [10]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/controller/AuthController.java

package com.example.wmspart3.controller;

import com.example.wmspart3.dto.SignupForm;
import com.example.wmspart3.service.UserService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;

@Controller
public class AuthController {

    private final UserService userService;

    public AuthController(UserService userService) {
        this.userService = userService;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/login";
    }

    @GetMapping("/login")
    public String login() {
        return "login";
    }

    @GetMapping("/signup")
    public String signupForm(Model model) {
        model.addAttribute("signupForm", new SignupForm());
        return "signup";
    }

    @PostMapping("/signup")
    public String signup(SignupForm form, Model model) {
        try {
            userService.signup(form);
            return "redirect:/login?signup";
        } catch (IllegalArgumentException e) {
            model.addAttribute("signupForm", form);
            model.addAttribute("errorMessage", e.getMessage());
            return "signup";
        }
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/controller/AuthController.java


In [11]:
%%writefile /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/controller/DashboardController.java

package com.example.wmspart3.controller;

import com.example.wmspart3.service.UserService;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.GetMapping;

@Controller
public class DashboardController {

    private final UserService userService;

    public DashboardController(UserService userService) {
        this.userService = userService;
    }

    @GetMapping("/customer/dashboard")
    public String customerDashboard(Authentication authentication, Model model) {
        model.addAttribute("email", authentication.getName());
        return "customer-dashboard";
    }

    @GetMapping("/admin/dashboard")
    public String adminDashboard(Authentication authentication, Model model) {
        model.addAttribute("email", authentication.getName());
        model.addAttribute("totalUsers", userService.countAllUsers());
        model.addAttribute("customerCount", userService.countCustomers());
        model.addAttribute("adminCount", userService.countAdmins());
        return "admin-dashboard";
    }
}

Writing /content/spring-lab/wms-part3/src/main/java/com/example/wmspart3/controller/DashboardController.java


In [12]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/templates/login.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>WMS 로그인</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="auth-page">
    <section class="auth-card">
        <p class="eyebrow">WMS PART III</p>
        <h1>로그인</h1>
        <p class="description">이메일과 비밀번호로 WMS에 접속한다.</p>

        <div class="message success" th:if="${param.signup}">
            회원가입이 완료되었습니다. 로그인하세요.
        </div>

        <div class="message error" th:if="${param.error}">
            이메일 또는 비밀번호가 올바르지 않습니다.
        </div>

        <div class="message error" th:if="${param.denied}">
            접근 권한이 없습니다.
        </div>

        <div class="message success" th:if="${param.logout}">
            로그아웃되었습니다.
        </div>

        <form method="post" action="/login">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label>
                이메일
                <input type="email" name="email" placeholder="user@example.com">
            </label>

            <label>
                비밀번호
                <input type="password" name="password" placeholder="비밀번호">
            </label>

            <button type="submit">로그인</button>
        </form>

        <div class="auth-link">
            계정이 없으면 <a href="/signup">회원가입</a>
        </div>
    </section>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part3/src/main/resources/templates/login.html


In [13]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/templates/signup.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>WMS 회원가입</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="auth-page">
    <section class="auth-card">
        <p class="eyebrow">WMS PART III</p>
        <h1>회원가입</h1>
        <p class="description">고객 또는 관리자 계정을 생성한다.</p>

        <div class="message error" th:if="${errorMessage != null}" th:text="${errorMessage}">
        </div>

        <form method="post" action="/signup" th:object="${signupForm}">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">

            <label>
                이메일
                <input type="email" th:field="*{email}" placeholder="user@example.com">
            </label>

            <label>
                비밀번호
                <input type="password" th:field="*{password}" placeholder="4자 이상">
            </label>

            <label>
                이름
                <input type="text" th:field="*{name}" placeholder="홍길동">
            </label>

            <label>
                사용자 유형
                <select th:field="*{role}">
                    <option value="">선택</option>
                    <option value="ROLE_CUSTOMER">고객</option>
                    <option value="ROLE_ADMIN">관리자</option>
                </select>
            </label>

            <button type="submit">회원가입</button>
        </form>

        <div class="auth-link">
            이미 계정이 있으면 <a href="/login">로그인</a>
        </div>
    </section>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part3/src/main/resources/templates/signup.html


In [14]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/templates/customer-dashboard.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>고객 대시보드</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar customer">
        <div class="brand">
            <div class="brand-mark">C</div>
            <div>
                <strong>WMS 고객</strong>
                <span>Customer</span>
            </div>
        </div>

        <nav class="menu">
            <a class="active">대시보드</a>
            <a>계약 현황</a>
            <a>입고 현황</a>
            <a>재고 현황</a>
            <a>출고 요청</a>
            <a>문의사항</a>
        </nav>

        <form method="post" action="/logout" class="logout-form">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
            <button type="submit">로그아웃</button>
        </form>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">CUSTOMER DASHBOARD</p>
            <h1>고객 대시보드</h1>
            <p th:text="${email} + ' 계정으로 로그인했습니다.'"></p>
        </section>

        <section class="summary-grid">
            <article>
                <span>내 계약</span>
                <strong>0</strong>
                <p>본인 계약 조회</p>
            </article>
            <article>
                <span>입고 현황</span>
                <strong>0</strong>
                <p>본인 물품 입고 상태</p>
            </article>
            <article>
                <span>재고 현황</span>
                <strong>0</strong>
                <p>본인 보관 재고</p>
            </article>
            <article>
                <span>출고 현황</span>
                <strong>0</strong>
                <p>본인 출고 요청</p>
            </article>
        </section>

        <section class="panel">
            <h2>고객 권한 확인</h2>
            <p>
                이 화면은 ROLE_CUSTOMER 권한을 가진 사용자만 접근할 수 있다.
                관리자 URL인 /admin/dashboard에 접근하면 권한 차단이 발생한다.
            </p>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part3/src/main/resources/templates/customer-dashboard.html


In [15]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/templates/admin-dashboard.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>관리자 대시보드</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="layout">
    <aside class="sidebar admin">
        <div class="brand">
            <div class="brand-mark">A</div>
            <div>
                <strong>WMS 관리자</strong>
                <span>Admin</span>
            </div>
        </div>

        <nav class="menu">
            <a class="active">대시보드</a>
            <a>계약 관리</a>
            <a>입고 관리</a>
            <a>재고 관리</a>
            <a>출고 관리</a>
            <a>공지사항</a>
            <a>문의사항</a>
            <a>사용자 관리</a>
        </nav>

        <form method="post" action="/logout" class="logout-form">
            <input type="hidden" th:name="${_csrf.parameterName}" th:value="${_csrf.token}">
            <button type="submit">로그아웃</button>
        </form>
    </aside>

    <main class="content">
        <section class="hero">
            <p class="eyebrow">ADMIN DASHBOARD</p>
            <h1>관리자 대시보드</h1>
            <p th:text="${email} + ' 관리자 계정으로 로그인했습니다.'"></p>
        </section>

        <section class="summary-grid">
            <article>
                <span>전체 사용자</span>
                <strong th:text="${totalUsers}">0</strong>
                <p>가입된 전체 계정</p>
            </article>
            <article>
                <span>고객 계정</span>
                <strong th:text="${customerCount}">0</strong>
                <p>ROLE_CUSTOMER</p>
            </article>
            <article>
                <span>관리자 계정</span>
                <strong th:text="${adminCount}">0</strong>
                <p>ROLE_ADMIN</p>
            </article>
            <article>
                <span>접근 제어</span>
                <strong>ON</strong>
                <p>Spring Security</p>
            </article>
        </section>

        <section class="panel">
            <h2>관리자 권한 확인</h2>
            <p>
                이 화면은 ROLE_ADMIN 권한을 가진 사용자만 접근할 수 있다.
                고객 계정으로 접근하면 Spring Security가 접근을 차단한다.
            </p>
        </section>
    </main>
</div>
</body>
</html>

Writing /content/spring-lab/wms-part3/src/main/resources/templates/admin-dashboard.html


In [16]:
%%writefile /content/spring-lab/wms-part3/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Noto Sans KR", sans-serif;
    background: #f5f7fb;
    color: #172033;
}

.auth-page {
    min-height: 100vh;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 40px;
}

.auth-card {
    width: 420px;
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 24px;
    padding: 34px;
    box-shadow: 0 16px 40px rgba(15, 23, 42, 0.08);
}

.eyebrow {
    margin: 0 0 10px;
    color: #1d4ed8;
    font-weight: 800;
    letter-spacing: 0.08em;
}

.auth-card h1,
.hero h1 {
    margin: 0 0 12px;
    font-size: 32px;
    letter-spacing: -0.04em;
}

.description,
.hero p {
    margin: 0 0 24px;
    color: #64748b;
    line-height: 1.7;
}

label {
    display: block;
    margin-bottom: 14px;
    font-weight: 800;
    color: #334155;
}

input,
select {
    width: 100%;
    margin-top: 8px;
    border: 1px solid #cbd5e1;
    border-radius: 12px;
    padding: 12px 13px;
    font-size: 14px;
    font-family: inherit;
    color: #0f172a;
    background: #ffffff;
}

button {
    border: none;
    border-radius: 12px;
    padding: 12px 14px;
    font-weight: 800;
    cursor: pointer;
    font-family: inherit;
    background: #1d4ed8;
    color: #ffffff;
}

.auth-card button {
    width: 100%;
    font-size: 15px;
}

.auth-link {
    margin-top: 18px;
    text-align: center;
    color: #64748b;
}

.auth-link a {
    color: #1d4ed8;
    font-weight: 800;
    text-decoration: none;
}

.message {
    border-radius: 14px;
    padding: 12px 14px;
    margin-bottom: 16px;
    font-weight: 800;
}

.message.error {
    background: #fff1f2;
    color: #be123c;
    border: 1px solid #fecdd3;
}

.message.success {
    background: #ecfdf3;
    color: #047857;
    border: 1px solid #bbf7d0;
}

.layout {
    display: flex;
    min-height: 100vh;
}

.sidebar {
    width: 260px;
    background: #ffffff;
    border-right: 1px solid #e2e8f0;
    padding: 28px 24px;
}

.sidebar.admin {
    background: #0f172a;
    color: #ffffff;
}

.sidebar.customer {
    background: #ffffff;
}

.brand {
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 42px;
}

.brand-mark {
    width: 44px;
    height: 44px;
    border-radius: 14px;
    background: #1d4ed8;
    color: #ffffff;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    font-size: 20px;
}

.sidebar.admin .brand-mark {
    background: #22c55e;
}

.brand strong {
    display: block;
    font-size: 19px;
}

.brand span {
    display: block;
    color: #64748b;
    font-size: 13px;
    margin-top: 3px;
}

.sidebar.admin .brand span {
    color: #94a3b8;
}

.menu {
    display: flex;
    flex-direction: column;
    gap: 10px;
}

.menu a {
    padding: 14px 16px;
    border-radius: 12px;
    font-weight: 700;
    color: #475569;
}

.sidebar.admin .menu a {
    color: #cbd5e1;
}

.menu a.active {
    background: #1d4ed8;
    color: #ffffff;
}

.sidebar.admin .menu a.active {
    background: #22c55e;
    color: #052e16;
}

.logout-form {
    margin-top: 30px;
}

.logout-form button {
    width: 100%;
    background: #ef4444;
}

.content {
    flex: 1;
    padding: 36px;
}

.hero {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 24px;
    padding: 34px;
    margin-bottom: 24px;
    box-shadow: 0 12px 30px rgba(15, 23, 42, 0.05);
}

.summary-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 18px;
    margin-bottom: 24px;
}

.summary-grid article {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 20px;
    padding: 24px;
}

.summary-grid span {
    color: #64748b;
    font-weight: 700;
}

.summary-grid strong {
    display: block;
    margin-top: 12px;
    font-size: 34px;
    color: #1d4ed8;
}

.summary-grid p {
    margin: 8px 0 0;
    color: #64748b;
    font-size: 14px;
}

.panel {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 20px;
    padding: 24px;
    box-shadow: 0 8px 24px rgba(15, 23, 42, 0.04);
}

.panel h2 {
    margin: 0 0 10px;
}

.panel p {
    margin: 0;
    color: #64748b;
    line-height: 1.7;
}

Writing /content/spring-lab/wms-part3/src/main/resources/static/style.css
